# Capítulo 3 — Temperatura, graus-dia e fenologia

**Curso:** Agrometeorologia Operacional com Python
**Prof. Dr. Fabrício Correia de Oliveira** — UTFPR, Campus Santa Helena

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fcoliveira-utfpr/agrometeorologia/blob/main/curso/03_temperatura_graus_dia.ipynb)

> Pré-requisito: Capítulos 1 e 2.

---


## 3.1 Motivação

Diferente do calendário civil, as plantas não "contam dias" — elas acumulam **calor**. Duas
safras semeadas na mesma data, em anos com temperaturas diferentes, atingem a maturação em
datas diferentes. O conceito de **graus-dia** formaliza essa ideia e permite prever, com boa
precisão, a data de colheita a partir apenas de temperatura e de duas constantes da cultivar.

Neste capítulo você vai implementar o cálculo de graus-dia e construir uma função genérica
que estima a data de maturação de **qualquer cultura**, a partir de **qualquer série real**
de temperatura — a mesma lógica usada depois no Capítulo 7 para definir janelas de semeadura.


## 3.2 Conceitos e fórmulas

- $T_b$ — temperatura basal inferior: abaixo dela o desenvolvimento da planta é nulo ou desprezível.
- $CT$ — constante térmica: total de graus-dia necessários para completar o ciclo.
- $NDA$ — número do dia do ano.

**Graus-dia diários:**
$$GD_i = \frac{T_{med} + T_{min}}{2} - T_b \qquad (\text{se } GD_i < 0,\ \text{adotar } GD_i = 0)$$

**Graus-dia acumulados no mês e no ciclo:**
$$GDA_{mes} = GD_i \times ND \qquad GDA_{ciclo} = \sum GDA_{mes}$$

**Dias necessários para atingir CT no mês final:**
$$n = \frac{CT - GDA_{ciclo\ anterior}}{GD_{i,\ mes\ final}}$$


## 3.3 Do papel ao código

Vamos implementar duas versões da mesma ideia:

1. `graus_dia_diario` — a fórmula pura, célula a célula, igual à apostila (útil para
   reproduzir o exercício resolvido e validar contra ele).
2. `estimar_data_maturacao` — uma versão **operacional**, que acumula graus-dia **dia a dia**
   a partir de uma série real de temperatura (por exemplo, o `df_clima` do Capítulo 1), sem
   depender de médias mensais fixas. É essa versão que reaproveitamos no resto do curso.

A `agrometeorologiapy` tem o equivalente no módulo `amp.grau_dias` — confira a assinatura
exata em `docs/FORMULAS.md` antes de trocar pela função da biblioteca.


In [ ]:
import pandas as pd
import numpy as np


def graus_dia_diario(Tmed: float, Tmin: float, Tb: float) -> float:
    """GDi — graus-dia acumulados em um único dia."""
    gd = (Tmed + Tmin) / 2 - Tb
    return max(gd, 0.0)


def estimar_data_maturacao(data_semeadura, Tb: float, CT: float, serie_temperatura: pd.DataFrame):
    """
    Estima a data de maturação acumulando graus-dia dia a dia.

    serie_temperatura: DataFrame indexado por data, com colunas 'T2M' (média) e 'T2M_MIN'.
    Retorna (data_maturacao, graus_dia_acumulados_por_dia: pd.Series).
    """
    serie = serie_temperatura.loc[serie_temperatura.index >= pd.Timestamp(data_semeadura)]
    gd_diario = serie.apply(lambda row: graus_dia_diario(row["T2M"], row["T2M_MIN"], Tb), axis=1)
    gd_acumulado = gd_diario.cumsum()

    dias_ate_maturacao = gd_acumulado[gd_acumulado >= CT]
    if dias_ate_maturacao.empty:
        return None, gd_acumulado  # a série não cobre dias suficientes
    data_maturacao = dias_ate_maturacao.index[0]
    return data_maturacao, gd_acumulado


## 3.4 Atividade guiada — reproduzindo o exercício do girassol

Semeadura de girassol em Marechal Cândido Rondon-PR em 25 de fevereiro, `Tb = 7,2 °C`,
`CT = 799 °C·d`. A apostila chega à maturação em **12 de maio**, usando temperaturas médias
mensais fixas. Vamos reproduzir o mesmo resultado primeiro pelo método da apostila (para
validar a fórmula) e depois discutir a diferença para o método dia a dia.


In [ ]:
# Reprodução fiel do exemplo da apostila (temperaturas médias mensais fixas)
Tb = 7.2
CT = 799

dados_mensais = pd.DataFrame({
    "mes": ["Fevereiro", "Março", "Abril", "Maio"],
    "Tmed": [27.2, 26.1, 23.0, 19.2],
    "Tmin": [17.1, 14.2, 10.5, 6.1],
    "ND":   [3, 31, 30, None],  # fevereiro: só 3 dias restantes após a semeadura em 25/fev
})
dados_mensais["GDi"] = dados_mensais.apply(lambda r: graus_dia_diario(r["Tmed"], r["Tmin"], Tb), axis=1)
dados_mensais


In [ ]:
gda_fev = dados_mensais.loc[0, "GDi"] * dados_mensais.loc[0, "ND"]
gda_mar = dados_mensais.loc[1, "GDi"] * dados_mensais.loc[1, "ND"]
gda_abr = dados_mensais.loc[2, "GDi"] * dados_mensais.loc[2, "ND"]
gda_ciclo_abr = gda_fev + gda_mar + gda_abr

faltam = CT - gda_ciclo_abr
gdi_maio = dados_mensais.loc[3, "GDi"]
n_dias = faltam / gdi_maio

print(f"GDA fev={gda_fev:.1f}  mar={gda_mar:.1f}  abr={gda_abr:.1f}  acumulado até abril={gda_ciclo_abr:.1f}")
print(f"Faltam {faltam:.1f} °C.d — a {gdi_maio:.2f} °C.d/dia → {n_dias:.0f} dias em maio")
print("Data de maturação estimada: 12 de maio (1º de maio + ~12 dias)")


## 3.5 Aplicando a versão operacional em dados reais

Agora vamos usar `estimar_data_maturacao`, que não depende de médias mensais fixas — ela
acumula graus-dia **dia a dia** a partir de uma série real. Isso é o que se usa na prática,
porque não exige assumir que a temperatura é constante dentro do mês.


In [ ]:
import requests

LAT, LON = -24.86, -54.33  # Santa Helena-PR
url = "https://power.larc.nasa.gov/api/temporal/daily/point"
params = {
    "parameters": "T2M,T2M_MAX,T2M_MIN,PRECTOTCORR,ALLSKY_SFC_SW_DWN,RH2M,WS2M",
    "community": "AG",
    "longitude": LON,
    "latitude": LAT,
    "start": "20230101",
    "end": "20231231",
    "format": "JSON",
}
resposta = requests.get(url, params=params, timeout=60)
resposta.raise_for_status()
propriedades = resposta.json()["properties"]["parameter"]

df_clima = pd.DataFrame(propriedades)
df_clima.index = pd.to_datetime(df_clima.index, format="%Y%m%d")
df_clima.index.name = "data"
df_clima = df_clima.replace(-999, np.nan)

# Exemplo: mesma cultivar de girassol (Tb=7.2, CT=799), semeadura em 25/fev/2023
data_maturacao, gd_acumulado = estimar_data_maturacao("2023-02-25", Tb=7.2, CT=799, serie_temperatura=df_clima)
print("Data de maturação estimada (dados reais de Santa Helena-PR, 2023):", data_maturacao)


A data pode diferir um pouco daquela obtida com médias mensais fixas — o método dia a dia
captura variações de curto prazo (uma semana mais fria ou mais quente) que a média mensal
esconde. Em aplicações operacionais reais, a versão dia a dia é a preferida.


## 3.6 Desafio

1. Repita a estimativa de maturação para pelo menos duas datas de semeadura diferentes
   (ex.: 25/fev e 15/mar) e compare as datas de maturação resultantes.
2. Escolha outra cultura com `Tb` e `CT` conhecidos (pesquise na literatura ou use valores
   típicos de milho: `Tb ≈ 10 °C`, `CT ≈ 1350 °C·d`) e estime a data de maturação para a
   sua região.
3. *(opcional)* Plote `gd_acumulado` ao longo do tempo, com uma linha horizontal em `CT`,
   para visualizar o momento em que a cultura atinge a maturação.


In [ ]:
# Espaço para o desafio — escreva seu código aqui


## 3.7 Checkpoint

Antes de seguir para o **Capítulo 4 — Umidade do ar e balanço de energia**, você deve ter:

- [ ] reproduzido o exercício do girassol com o mesmo resultado da apostila (12 de maio);
- [ ] uma função `estimar_data_maturacao` funcionando com dados reais;
- [ ] resolvido pelo menos o item 1 do desafio.
